# Sentinel-2: exploração futura
A floração dos ipês urbanos produz uma assinatura detectável nas séries temporais Sentinel-2?

Base sem resultados. Configure uma árvore confirmada; não execute com coordenadas inventadas. Uma copa pode ser menor que um pixel de 10 m. NDVI não é prova de floração.

In [ ]:
EXECUTAR = False
ARVORE_ID = None
LATITUDE = None
LONGITUDE = None
PERIODO = None  # intervalo ISO: inicio/fim, escolhido após revisão


In [ ]:
if EXECUTAR:
    assert ARVORE_ID and LATITUDE is not None and LONGITUDE is not None and PERIODO
    from pystac_client import Client
    import planetary_computer as pc
    import stackstac
    import numpy as np
    import pandas as pd
    catalog = Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=pc.sign_inplace)
    bbox = [LONGITUDE - 0.001, LATITUDE - 0.001, LONGITUDE + 0.001, LATITUDE + 0.001]
    search = catalog.search(collections=['sentinel-2-l2a'], bbox=bbox, datetime=PERIODO, query={'eo:cloud_cover': {'lt': 30}}, max_items=100)
    items = list(search.items())
    print(f'{len(items)} cenas encontradas; máximo 100, refine o intervalo se necessário.')


In [ ]:
if EXECUTAR and items:
    # Goiânia: UTM 22S. Fora desta região, escolher o CRS apropriado.
    cube = stackstac.stack(items, assets=['B04', 'B08', 'SCL'], epsg=32722, resolution=10, bounds_latlon=bbox, rescale=False)
    red = cube.sel(band='B04').astype('float32')
    nir = cube.sel(band='B08').astype('float32')
    scl = cube.sel(band='SCL')
    clear = scl.isin([4, 5, 6])  # máscara conservadora: vegetação, não vegetado, água
    # Reflectância: usar escala/offset declarados pelo asset STAC.
    scales = {}
    for band in ['B04', 'B08']:
        calibration = items[0].assets[band].extra_fields.get('raster:bands', [{}])[0]
        assert 'scale' in calibration, 'Confira escala e offset da coleção antes de calcular NDVI'
        signatures = {(i.assets[band].extra_fields.get('raster:bands', [{}])[0].get('scale'), i.assets[band].extra_fields.get('raster:bands', [{}])[0].get('offset', 0)) for i in items}
        assert len(signatures) == 1, 'Calibração varia entre cenas; aplicar por cena antes da análise'
        scales[band] = (calibration['scale'], calibration.get('offset', 0))
    red = red * scales['B04'][0] + scales['B04'][1]
    nir = nir * scales['B08'][0] + scales['B08'][1]
    ndvi = ((nir - red) / (nir + red)).where(clear & ((nir + red) != 0))


In [ ]:
if EXECUTAR and items:
    from pyproj import Transformer
    x, y = Transformer.from_crs(4326, 32722, always_xy=True).transform(LONGITUDE, LATITUDE)
    pixel = ndvi.sel(x=x, y=y, method='nearest').compute()
    frame = pd.DataFrame({'data': pd.to_datetime(pixel.time.values), 'ndvi': np.asarray(pixel.values), 'arvore_id': ARVORE_ID})
    frame.plot(x='data', y='ndvi', marker='o', title='Exploração de pixel misto — sem inferência de floração')


## Antes de interpretar
Compare com as datas fenológicas confirmadas, documente cenas excluídas e nuvens, valide escala/offset por cena, avalie pixels vizinhos e controles. Nenhum resultado deste notebook comprova uma espécie ou detecção automática. Não publique coordenadas privadas nem resultados sem revisão.